# Redrob Hackathon — Phase 2 Sandbox
**Team:** Mayank_kumar | **Top-170 input → Top-25 output**

This notebook runs the complete Phase 2 ranking pipeline on the top-170 candidates from Phase 1 and returns the top-25 ranked candidates.
No Google Drive access needed — all artifacts are in the repo.
--Click on Run All icon on the top. You will see the output csv file.

## Step 1 — Install dependencies

In [ ]:
!pip install polars faiss-cpu sentence-transformers numpy -q
print('Dependencies installed.')

## Step 2 — Clone repository (with Git LFS artifacts)

In [ ]:
import os

# Install Git LFS if not present
!apt-get install -y git-lfs -q
!git lfs install

REPO = 'https://github.com/Mayankkumar24/Redrob_Assessment.git'
DEST = '/content/Redrob_Assessment'

if os.path.exists(DEST):
    print('Repo already cloned — pulling latest changes.')
    os.chdir(DEST)
    !git pull
else:
    !git clone {REPO} {DEST}
    os.chdir(DEST)

import sys
if DEST not in sys.path:
    sys.path.insert(0, DEST)

print('Working directory:', os.getcwd())
print('Repo structure:')
!ls -1

## Step 3 — Verify sandbox artifacts

In [ ]:
import os

required = [
    'data/processed/top170_candidates.json',
    'data/sandbox_artifacts/candidates_clean.parquet',
    'data/sandbox_artifacts/candidate_embeddings.npy',
    'data/sandbox_artifacts/candidate_ids.npy',
]

all_ok = True
for path in required:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = f'✓  {size:>7.1f} KB' if exists else '✗  MISSING'
    print(f'{status}  {path}')
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        'Some artifacts are missing. Make sure Git LFS pulled correctly.\n'
        'Run: git lfs pull'
    )
print('\nAll artifacts present.')

## Step 4 — Point config to sandbox artifacts

In [ ]:
from pathlib import Path
from ranker import config

SANDBOX = Path('/content/Redrob_Assessment/data/sandbox_artifacts')

# Override paths to use sandbox artifacts (170-candidate subset)
config.PATHS['candidate_meta'] = SANDBOX / 'candidates_clean.parquet'
config.PATHS['candidate_emb']  = SANDBOX / 'candidate_embeddings.npy'
config.PATHS['candidate_ids']  = SANDBOX / 'candidate_ids.npy'
# top_170.json stays as-is (already in ranker/)

print('Paths configured:')
for k, v in config.PATHS.items():
    print(f'  {k}: {v}')

## Step 5 — Run Phase 2 ranking pipeline

In [ ]:
# Reload modules so config overrides take effect
import importlib
import ranker.config, ranker.embed, ranker.scorer
import ranker.behavioral, ranker.skills, ranker.reasoning, ranker.main

for mod in [
    ranker.config, ranker.skills, ranker.behavioral,
    ranker.scorer, ranker.reasoning, ranker.embed, ranker.main
]:
    importlib.reload(mod)

from ranker.main import run

out_path = run(
    output_path=Path('/content/Redrob_Assessment/sandbox_output.csv'),
    top_k=25,       # sandbox shows top-25 only
    verbose=True,
)
print(f'\nOutput written to: {out_path}')

## Step 6 — Results

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Redrob_Assessment/sandbox_output.csv')

print(f'Total ranked candidates: {len(df)}')
print(f'Score range: {df["score"].min():.4f} — {df["score"].max():.4f}')
print()

# Display full table
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 25)
display(df[['candidate_id', 'rank', 'score', 'reasoning']])

## Step 7 — Download output CSV

In [ ]:
try:
    from google.colab import files
    files.download('/content/Redrob_Assessment/sandbox_output.csv')
    print('Download started.')
except ImportError:
    import os
    path = os.path.abspath('Mayank_kumar_sandbox_output.csv')
    print(f'Running locally — file saved at: {path}')